<a href="https://colab.research.google.com/github/Jamalbah64/bone-fracture-learning-portal/blob/Jamal-branch/FractureClassifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Preparing the Notebook

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Cell 2 - Force local ultralytics
import sys
!pip uninstall ultralytics -y
for key in list(sys.modules.keys()):
    if 'ultralytics' in key.lower():
        del sys.modules[key]
sys.path.insert(0, '/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8')

import ultralytics
print(ultralytics.__file__)
# Must show FracDetectYolo path, NOT /usr/local/lib

/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/ultralytics/__init__.py


In [ ]:
# Cell 3 - Install dependencies
!pip install timm tqdm scipy seaborn matplotlib psutil py-cpuinfo requests scikit-optimize

In [ ]:
# Patch torch.load to use weights_only=False for compatibility

import torch
original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = patched_load
print("Torch load patched successfully")

Torch load patched successfully


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## Preparing the Dataset

### Filestems Query

This section of the notebook uses the query to get the important filestems for model_u, model_a, and model_b.



In [ ]:
# Split Data and get Data Frame
from __future__ import annotations

from pathlib import Path
from typing import Final

import pandas as pd
from sqlalchemy import (
    MetaData,
    Table,
    and_,
    cast,
    create_engine,
    func,
    literal,
    select,
    String,
)
from sqlalchemy.engine import Engine

# Internal sentinel for empty ao_classification values.
_NULL_CLASS: Final[str] = ""


class ModelDataQuery:
    def __init__(self, dataset_csv: Path | str) -> None:
        self.dataset_csv = Path(dataset_csv)

        if not self.dataset_csv.exists():
            raise FileNotFoundError(f"Could not find dataset CSV at: {self.dataset_csv}")

        self.engine: Engine = create_engine("sqlite+pysqlite:///:memory:", future=True)
        self.metadata = MetaData()

        self._load_csv()
        self.dataset = Table("dataset", self.metadata, autoload_with=self.engine)

    def _load_csv(self) -> None:
        dataframe = pd.read_csv(self.dataset_csv)
        dataframe.columns = [column.strip() for column in dataframe.columns]
        dataframe.to_sql("dataset", self.engine, if_exists="replace", index=False)

    def _get_model_data(
        self,
        projections: list[int],
        min_class_size: int,
        max_class_count: int,
    ) -> list[str]:
        if not projections:
            raise ValueError("projections must contain at least one projection value")

        if min_class_size < 1:
            raise ValueError("min_class_size must be at least 1")

        if max_class_count < 1:
            raise ValueError("max_class_count must be at least 1")

        dataset = self.dataset

        filestem_expr = cast(dataset.c.filestem, String)
        projection_expr = cast(dataset.c.projection, String)
        class_raw_expr = cast(dataset.c.ao_classification, String)

        trimmed_filestem = func.trim(filestem_expr)
        trimmed_class = func.trim(class_raw_expr)

        class_name_expr = func.coalesce(
            func.nullif(trimmed_class, ""),
            literal(_NULL_CLASS),
        ).label("class_name")

        normalized = (
            select(
                filestem_expr.label("filestem"),
                projection_expr.label("projection"),
                class_name_expr,
            )
            .where(
                and_(
                    dataset.c.filestem.is_not(None),
                    trimmed_filestem != "",
                    projection_expr.in_([str(projection) for projection in projections]),
                )
            )
            .cte("normalized")
        )

        eligible_classes = (
            select(
                normalized.c.class_name,
                func.count().label("class_count"),
            )
            .group_by(normalized.c.class_name)
            .having(func.count() >= min_class_size)
            .order_by(
                func.count().desc(),
                normalized.c.class_name.asc(),
            )
            .limit(max_class_count)
            .cte("eligible_classes")
        )

        query = (
            select(normalized.c.filestem)
            .join(
                eligible_classes,
                normalized.c.class_name == eligible_classes.c.class_name,
            )
            .order_by(
                eligible_classes.c.class_count.desc(),
                normalized.c.class_name.asc(),
                normalized.c.filestem.asc(),
            )
        )

        with self.engine.connect() as connection:
            result = connection.execute(query)
            return [str(row[0]) for row in result.fetchall()]

    def get_modelA_data(self, min_class_size: int, max_class_count: int) -> list[str]:
        return self._get_model_data(
            projections=[1],
            min_class_size=min_class_size,
            max_class_count=max_class_count,
        )

    def get_modelB_data(self, min_class_size: int, max_class_count: int) -> list[str]:
        return self._get_model_data(
            projections=[2],
            min_class_size=min_class_size,
            max_class_count=max_class_count,
        )

    def get_modelU_data(self, min_class_size: int, max_class_count: int) -> list[str]:
        return self._get_model_data(
            projections=[1, 2],
            min_class_size=min_class_size,
            max_class_count=max_class_count,
        )

query = ModelDataQuery("/content/drive/MyDrive/FracDetectYolo/filtered_dataset/full_dataset/dataset.csv")

model_b_stems = query.get_modelB_data(min_class_size=10, max_class_count=9)

print(f"Length of Model B List: {len(model_b_stems)}")

print("\nFirst 10 Model A stems:")
for stem in model_b_stems[:10]:
    print(stem)


FileNotFoundError: Could not find dataset CSV at: /content/drive/MyDrive/FracDetectYolo/filtered_dataset/full_dataset/dataset.csv

In [ ]:
# Check Classes in Model Stems
import os
import pandas as pd
from collections import Counter

csv_path  = '/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/GRAZPEDWRI-DX/dataset.csv'
labels_dir = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/labels_model_b'

df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
ao_lookup = dict(zip(df['filestem'], df['ao_classification']))

ao_counter = Counter()
empty_count = 0

for stem in model_b_stems:
    ao_raw = ao_lookup.get(stem)
    if ao_raw is None or pd.isna(ao_raw):
        empty_count += 1
    else:
        ao_counter[str(ao_raw).strip()] += 1

print(f"All AO classifications in model_b_stems:\n")
print(f"  Non-fracture: {empty_count}")
for ao, count in sorted(ao_counter.items(), key=lambda x: x[1], reverse=True):
    print(f"  {ao}: {count}")
print(f"\nTotal unique AO codes: {len(ao_counter)}")
print(f"Total stems: {len(model_b_stems)}")

### Remapping and Copying Labels

Labels need to be remapped and copied before they can be used. Our model is going to expect specific classes to be labeled a certain way, and the raw dataset labels them differently.

In [ ]:
# Get and Copy Needed Labels for the Model
import os
import shutil

labels_source = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/labels'
labels_output = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/labels_model_a'

os.makedirs(labels_output, exist_ok=True)

copied  = 0
missing = 0

for stem in model_a_stems:
    src = os.path.join(labels_source, f"{stem}.txt")
    dst = os.path.join(labels_output, f"{stem}.txt")

    if os.path.exists(src):
        shutil.copy2(src, dst)
        copied += 1
    else:
        print(f"  MISSING: {stem}")
        missing += 1

    if(copied % 200 == 0):
      print(f"Copied {copied}/{len(model_a_stems)} labels")

print(f"\nDone — {copied} copied, {missing} missing")

In [ ]:
# Remap Labels
import os
import pandas as pd
from enum import IntEnum

csv_path     = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/dataset.csv'
labels_input  = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/labels_model_a'
labels_output = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/labels_model_a_remapped'

os.makedirs(labels_output, exist_ok=True)

# Set Classes to be Remapped
target_classes = [
    '23r-M/2.1',   # 0
    '23r-M/3.1',   # 1
    '23-M/3.1',    # 2
    '23-M/2.1',    # 3
    '23r-E/2.1',   # 4
    '22r-D/2.1',   # 5
    '23r-E/1',     # 6
    '22-D/2.1',    # 7
]

# Build AO codes
ao_to_id = {ao: idx for idx, ao in enumerate(target_classes)}

# Load CSV for AO lookup
df        = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
ao_lookup = dict(zip(df['filestem'], df['ao_classification']))

remapped     = 0
non_fracture = 0
skipped      = 0
dropped      = 0

for stem in model_a_stems:
    src    = os.path.join(labels_input,  f"{stem}.txt")
    dst    = os.path.join(labels_output, f"{stem}.txt")
    ao_raw = ao_lookup.get(stem)

    # Non-fracture: empty label file
    if ao_raw is None or pd.isna(ao_raw):
        open(dst, 'w').close()
        non_fracture += 1
        continue

    # Look up new class ID from target list
    ao_str   = str(ao_raw).strip()
    new_id   = ao_to_id.get(ao_str)

    # AO code not in target classes -> skip
    if new_id is None:
        dropped += 1
        continue

    if not os.path.exists(src):
        print(f"  MISSING: {stem}")
        skipped += 1
        continue

    with open(src, 'r') as f:
      lines = [l.strip() for l in f.readlines() if l.strip()]

    if not lines:
        print(f"  EMPTY LABEL (has AO code): {stem}: '{ao_str}'")
        skipped += 1
        continue

    # Find where fracture is in label (Line starts with 3)
    fracture_line = None
    for line in lines:
        if int(line.split()[0]) == 3:
          fracture_line = line
          break

    # No line exists for a fracutre box (Fracture not visible but classified)
    # Skip label and don't add to new dataset
    if fracture_line is None:
      print(f" AO Code but No Fracture Box: {stem}: '{ao_str}'")
      skipped += 1
      continue

    rest = fracture_line.split(' ', 1)[1]

    with open(dst, 'w') as f:
        f.write(f"{new_id} {rest}\n")

    remapped += 1

    if(remapped % 200 == 0):
      print(f"Remapped {remapped} labels")

print(f"\nDone!")
print(f"  Remapped:          {remapped}")
print(f"  Non-fracture:      {non_fracture}")
print(f"  Dropped (not in target classes): {dropped}")
print(f"  Missing files:     {skipped}")

In [ ]:
#Rebuild Model Stems with New Labels
import os

labels_dir = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/labels_model_b_remapped'

# Rebuild model_a_stems from what actually exists and has content
model_b_stems = []
for fname in os.listdir(labels_dir):
    if not fname.endswith('.txt'):
        continue
    stem = os.path.splitext(fname)[0]
    model_b_stems.append(stem)

print(f"Updated model_a_stems: {len(model_b_stems)} stems")

### Splitting and Augmenting Data

In [ ]:
#Cap Fractures to 1000 images
from collections import defaultdict
import random

random.seed(42)

csv_path = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/dataset.csv'

df        = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
ao_lookup = dict(zip(df['filestem'], df['ao_classification']))

# Group stems by AO classification
ao_to_stems = defaultdict(list)
for stem in model_b_stems:
    ao = ao_lookup.get(stem)
    if pd.isna(ao):
        ao_to_stems['non_fracture'].append(stem)
    else:
        ao_to_stems[str(ao).strip()].append(stem)

# Cap each class at 1000
CAP = 1000
capped_stems = []
for ao, stems in ao_to_stems.items():
    if len(stems) > CAP:
        capped_stems.extend(random.sample(stems, CAP))
    else:
        capped_stems.extend(stems)

print(f"Total before cap: {len(model_b_stems)}")
print(f"Total after cap:  {len(capped_stems)}")
print()
for ao, stems in sorted(ao_to_stems.items(), key=lambda x: len(x[1]), reverse=True):
    actual = min(len(stems), CAP)
    print(f"  {ao}: {len(stems)} -> {actual}")

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import defaultdict
import random

random.seed(42)

df = pd.read_csv('/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/GRAZPEDWRI-DX/dataset.csv')
df.columns = [c.strip() for c in df.columns]
stem_to_patient = dict(zip(df['filestem'], df['patient_id']))

patient_to_stems = defaultdict(list)
for stem in capped_stems:
    # Strip augmentation suffix if present to get base stem for patient lookup
    base_stem  = '_aug_'.join(stem.split('_aug_')[:-1]) if '_aug_' in stem else stem
    patient_id = stem_to_patient.get(base_stem)
    if patient_id:
        patient_to_stems[patient_id].append(stem)

# Split at patient level
patients = list(patient_to_stems.keys())

train_patients, temp_patients = train_test_split(patients, test_size=0.30, random_state=42)
val_patients, test_patients   = train_test_split(temp_patients, test_size=0.333, random_state=42)

train_stems = [s for p in train_patients for s in patient_to_stems[p]]
val_stems   = [s for p in val_patients   for s in patient_to_stems[p]]
test_stems  = [s for p in test_patients  for s in patient_to_stems[p]]

print(f"Train: {len(train_stems)}  Val: {len(val_stems)}  Test: {len(test_stems)}")

In [ ]:
#Checks Training Set Distribution
import os
from collections import Counter

labels_dir = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/labels_model_b_remapped'

target_classes = [
    '23r-M/2.1',   # 0
    '23r-M/3.1',   # 1
    '23-M/3.1',    # 2
    '23-M/2.1',    # 3
    '23r-E/2.1',   # 4
    '22r-D/2.1',   # 5
    '23r-E/1',     # 6
    '22-D/2.1',    # 7
]

class_counter = Counter()
empty_count = 0

for stem in train_stems:
    ao = ao_lookup.get(stem)
    if ao is None or ao == 'Non-fracture':
        empty_count += 1
    else:
        # map AO code back to class id
        ao_to_class = {name: i for i, name in enumerate(target_classes)}
        class_id = ao_to_class.get(ao)
        if class_id is not None:
            class_counter[class_id] += 1

print(f"Train set class distribution:")
print(f"  Non-fracture (empty): {empty_count}")
for cls_id in sorted(class_counter):
    name = target_classes[cls_id] if cls_id < len(target_classes) else f'class_{cls_id}'
    print(f"  Class {cls_id} ({name}): {class_counter[cls_id]}")
print(f"\nTotal fracture stems: {sum(class_counter.values())}")
print(f"Total train stems: {len(train_stems)}")

In [ ]:
# Data Augmentation
import os  # for file handling
import cv2  # for image processing
import albumentations as A  # for data augmentation

class_aug_counts = {
    4: 3,    # 23r-E/2.1: 154 × 4 = 744
    5: 9,    # 22r-D/2.1:  25 × 10 = 250
    6: 9,    # 23r-E/1:    17 × 10 = 170
    7: 9,    # 22-D/2.1:   23 × 10 = 230
}

# Main functions

def load_yolo_labels(label_path):
    """
    This function reads a YOLO format label
    and extracts the class labels and bounding boxes.
    Args:
    label_path (str): The path to the label file in YOLO format
    Returns:
    class_labels (list of int): A list of class labels corresponding to the bounding boxes
    bboxes (list of tuples): A list of bounding boxes in YOLO format (x_center, y_center, width, height)
    """
    bboxes = []  # list to store bounding boxes
    class_labels = []  # list to store class labels

    if not os.path.exists(label_path):  # check if label file exists
        return class_labels, bboxes

    with open(label_path, 'r') as f:  # open the label file for reading
        for line in f:  # reads each line in the file
            parts = line.strip().split()
            if len(parts) != 5:  # if the line does not have exactly 5 parts, skip it
                continue
            class_id = int(parts[0])  # class id part of the line
            x_center = float(parts[1])  # x center of the bbox
            y_center = float(parts[2])  # y center of the bbox
            width = float(parts[3])  # width of the bbox
            height = float(parts[4])  # height of the bbox

            class_labels.append(class_id)
            bboxes.append((x_center, y_center, width, height))

    return class_labels, bboxes

def save_yolo_labels(label_path, class_labels, bboxes):
    """
    This function saves the class labels
    and bounding boxes in YOLO format to a label file.

    Args:
    label_path (str): The path to the label file where the augmented labels will be saved
    class_labels (list of int): A list of class labels corresponding to the bounding boxes
    bboxes (list of tuples): A list of bounding boxes in YOLO format (x_center, y_center, width, height)
    """
    with open(label_path, 'w') as f:  # open the label file for writing
        for class_id, bbox in zip(class_labels, bboxes):
            x_center, y_center, width, height = bbox
            # write the class id and bbox to the file
            print(
                f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}", file=f)


def get_augmentation_pipeline():
    """
    This function defines the augmentation pipeline using albumentations.
    It includes horizontal flips, rotations, affine transformations,
    and brightness/contrast adjustments.

    Args:
    Returns:
        A.Compose: The defined augmentation pipeline.
    """
    return A.Compose(
        [
            # A.HorizontalFlip(p=0.5),
            # # Border mode replicate to avoid black borders after rotation
            # A.Rotate(limit=10, border_mode=cv2.BORDER_REPLICATE, p=0.5),
            A.RandomBrightnessContrast(  # adjust brightness and contrast to simulate different lighting conditions
                brightness_limit=0.2,  # avoid making the image too dark or too bright
                contrast_limit=0.2,  # avoids drastic contrast changes
                p=0.8
            ),
            # apply Gaussian blur to simulate motion blur or out-of-focus images
            A.GaussianBlur(blur_limit=(3, 7), p=0.3),
        ],
        bbox_params=A.BboxParams(  # specify the bounding boxes in YOLO format and how to handle them during augmentation
            format="yolo",
            label_fields=["class_labels"],
            min_visibility=0.3  # only keep boxes with 30% visibility after augmentation
        )
    )


def augment_image_and_boxes(image, bboxes, class_labels, transform):
    """
    This function applies the augmentation pipeline to an image
    and its corresponding bounding boxes and class labels.
    It returns the augmented image, bounding boxes, and class labels.

    Args:
    image (numpy array): The input image to be augmented.
    bboxes (list of tuples): A list of bounding boxes in YOLO format (x_center, y_center, width, height).
    class_labels (list of int): A list of class labels corresponding to the bounding boxes.
    transform (albumentations.Compose): The augmentation pipeline.
    """
    augmented = transform(
        image=image,
        bboxes=bboxes,
        class_labels=class_labels
    )

    # Extract the augmented image, bounding boxes, and class labels from the augmented result
    aug_image = augmented["image"]
    aug_bboxes = augmented["bboxes"]
    aug_class_labels = augmented["class_labels"]
    return aug_image, aug_bboxes, aug_class_labels


def process_train_folder(images_dir, labels_dir, stems, num_augments=2):
    """
    process_train_folder is the main function that processes the training folder
    by applying augmentations to the images and their corresponding labels.

    It takes in the directory paths for images and labels, as well as the number of augmentations to apply to each image.
    The function reads each image and its corresponding label, applies the defined augmentation pipeline,
    and saves the augmented images and labels in a new directory structure.

    Args:
        images_dir (_type_): _description_
        labels_dir (_type_): _description_
        num_augments (int, optional): The number of augmentations to apply to each image. Defaults to 2.
    """

    # get the defined augmentation pipeline
    transform = get_augmentation_pipeline()

    if not os.path.exists(images_dir):
        print(f"Error: Images directory not found: {images_dir}")
        return

    if not os.path.exists(labels_dir):
        print(f"Error: Labels directory not found: {labels_dir}")
        return

    valid_exts = (".png", ".jpg", ".jpeg")  # valid image extensions

    augmented_images = 0
    MINORITY_CLASSES = {4, 5, 6, 7}
    for stem in stems:
        # iterate through image files
        # gets the full path of the image
        image_path = os.path.join(images_dir, f"{stem}.png")
        label_path = os.path.join(labels_dir, stem + ".txt")

        # cv2 is used to read the image from the path
        image = cv2.imread(image_path)
        if image is None:  # if the image cannot be read, skip it
            print(f"Warning: Could not read image {image_path}. Skipping.")
            continue

        # load the corresponding labels for the image
        class_labels, bboxes = load_yolo_labels(label_path)

        if not class_labels or class_labels[0] not in MINORITY_CLASSES:
          continue

        # Skip unlabeled images for positive fracture augments
        # if len(bboxes) == 0:
        #     continue

        class_id = class_labels[0]
        num_augments = class_aug_counts.get(class_id, 0)

        for i in range(num_augments):  # apply augmentations to the image and labels
            try:
                aug_image, aug_bboxes, aug_class_labels = augment_image_and_boxes(
                    image, bboxes, class_labels, transform)

                # Save the augmented image and labels

                # create a new name for the augmented image
                aug_image_name = f"{stem}_aug_{i}.jpg"
                # create a new name for the augmented label
                aug_label_name = f"{stem}_aug_{i}.txt"

                # get the path to save the augmented image
                aug_image_path = os.path.join('/content/model_b_aug/images', aug_image_name)
                # get the path to save the augmented label
                aug_label_path = os.path.join('/content/model_b_aug/labels', aug_label_name)

                # save the augmented image to disk
                cv2.imwrite(aug_image_path, aug_image)
                # save the augmented labels to disk
                save_yolo_labels(aug_label_path, aug_class_labels, aug_bboxes)

                augmented_images += 1

                if(augmented_images % 200 == 0):
                  print(f"Augmented {augmented_images}/1047 images")

            except Exception as e:
                print(f"Failed on {stem} with error: {e}")

os.makedirs('/content/model_b_aug/images', exist_ok=True)
os.makedirs('/content/model_b_aug/labels', exist_ok=True)

process_train_folder(
        images_dir='/content/drive/MyDrive/FracDetectYolo/filtered_dataset/full_dataset',
        labels_dir='/content/drive/MyDrive/FracDetectYolo/filtered_dataset/labels_model_b_remapped',
        stems=train_stems
    )

print("Augmentation complete.")

#Add Augmented Stems to Training Stems
aug_stems = [
    os.path.splitext(f)[0]
    for f in os.listdir('/content/model_b_aug/labels')
    if f.endswith('.txt')
]

train_stems = train_stems + aug_stems
print(f"Train stems after augmentation: {len(train_stems)}")

In [ ]:
from pathlib import Path
path_images = Path("/content/model_b_aug/images")
# Only count items that are files
count_images = len([f for f in path_images.iterdir() if f.is_file()])
print(f"{count_images} images ")

path_labels = Path("/content/model_b_aug/labels")
# Only count items that are files
count_labels = len([f for f in path_labels.iterdir() if f.is_file()])
print(f"{count_labels} labels ")


### Writing Over Images and Validation

In [ ]:
#Copy Over Images with their Splits
import os
import shutil

images_dir = "/content/drive/MyDrive/FracDetectYolo/filtered_dataset/full_dataset"
labels_dir = "/content/drive/MyDrive/FracDetectYolo/filtered_dataset/labels_model_b_remapped"
aug_img_dir  = "/content/model_b_aug/images"
aug_lbl_dir  = "/content/model_b_aug/labels"

base_images = "/content/model_b/images"
base_labels = "/content/model_b/labels"

for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(base_images, split), exist_ok=True)
    os.makedirs(os.path.join(base_labels, split), exist_ok=True)

#Check all Labels/Images before
print("Running Pre-Check")
all_stems = train_stems + val_stems + test_stems

missing_images = []
missing_labels = []

for stem in all_stems:
    is_aug  = '_aug_' in stem
    img_src = os.path.join(aug_img_dir if is_aug else images_dir, f"{stem}.{'jpg' if is_aug else 'png'}")
    lbl_src = os.path.join(aug_lbl_dir if is_aug else labels_dir, f"{stem}.txt")

    if not os.path.exists(img_src):
        missing_images.append(stem)
    if not os.path.exists(lbl_src):
        missing_labels.append(stem)

print(f"  Total stems:    {len(all_stems)}")
print(f"  Missing images: {len(missing_images)}")
print(f"  Missing labels: {len(missing_labels)}")

if missing_images or missing_labels:
    print("\nMissing files found — skipping those stems during copy")
else:
    print("\nAll files accounted for — proceeding with copy")

# Copy and Skip Missing
def copy_split(stems, split_name):
    copied  = 0
    skipped = 0
    for stem in stems:
        is_aug  = '_aug_' in stem
        ext     = 'jpg' if is_aug else 'png'
        img_src = os.path.join(aug_img_dir if is_aug else images_dir, f"{stem}.{ext}")
        lbl_src = os.path.join(aug_lbl_dir if is_aug else labels_dir, f"{stem}.txt")
        img_dst = os.path.join(base_images, split_name, f"{stem}.{ext}")
        lbl_dst = os.path.join(base_labels, split_name, f"{stem}.txt")

        if not os.path.exists(img_src) or not os.path.exists(lbl_src):
            skipped += 1
            continue

        shutil.copy2(img_src, img_dst)
        shutil.copy2(lbl_src, lbl_dst)
        copied += 1

        if copied % 200 == 0:
            print(f"  {split_name}: {copied}/{len(stems)} copied")

    print(f"{split_name}: {copied} copied, {skipped} skipped")

copy_split(train_stems, 'train')
copy_split(val_stems,   'val')
copy_split(test_stems,  'test')

In [ ]:
#Write Meta.yaml so model knows where images are
import yaml

meta = {
    'path':  '/content/model_b/',
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc':    8,
    'names': {
        0: '23r-M/2.1',
        1: '23r-M/3.1',
        2: '23-M/3.1',
        3: '23-M/2.1',
        4: '23r-E/2.1',
        5: '22r-D/2.1',
        6: '23r-E/1',
        7: '22-D/2.1',
    }
}

meta_path = '/content/model_b/meta.yaml'
with open(meta_path, 'w') as f:
    yaml.dump(meta, f, default_flow_style=False, allow_unicode=True)

print(f"Written meta.yaml with nc={meta['nc']}")
for idx, name in meta['names'].items():
    print(f"  {idx}: {name}")

In [ ]:
#Verify meta.yml
with open('/content/model_a/meta.yaml', 'r') as f:
    print(f.read())

In [ ]:
#Check Train Images
import os

base = '/content/model_b'
for folder in ['images/train', 'images/val', 'images/test',
               'labels/train', 'labels/val', 'labels/test']:
    path = os.path.join(base, folder)
    if os.path.exists(path):
        print(f"{folder}: {len(os.listdir(path))} files")
    else:
        print(f"{folder}: NOT FOUND")

## Single Split Training and Testing

In [ ]:
#Train Model
from ultralytics import YOLO
from ultralytics.utils import callbacks
import shutil
import os

backup_dir = '/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_b_augv2_backup/checkpoints'
os.makedirs(backup_dir, exist_ok=True)

def backup_on_epoch_end(trainer):
    epoch = trainer.epoch
    if epoch % 5 == 0 or epoch == trainer.epochs - 1:
        try:
            shutil.copy2(str(trainer.best), f"{backup_dir}/epoch_{epoch}_best.pt")
            results_csv = 'fracture_detection/model_b_augv2/results.csv'
            if os.path.exists(results_csv):
                shutil.copy2(results_csv, f"{backup_dir}/results_epoch_{epoch}.csv")
            print(f"Backup saved at epoch {epoch}")
        except Exception as e:
            print(f"Backup failed at epoch {epoch}: {e}")

model = YOLO('/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/ultralytics/cfg/models/v8/yolov8m_SE_M3.yaml')

model.add_callback('on_fit_epoch_end', backup_on_epoch_end)

model.train(
    data='/content/model_b/meta.yaml',
    epochs=100,
    patience=20,
    batch=16,
    optimizer='SGD',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.001,
    warmup_epochs=5,
    cos_lr=True,
    cls=1.5,
    label_smoothing=0.1,
    device=0,
    project='fracture_detection',
    name='model_b_augv2',
    exist_ok=True,

    # Augmentation disabled
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
)

shutil.copytree(
    '/content/fracture_detection/model_b_augv2',
    '/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_b_augv2',
    dirs_exist_ok=True
)
print("Done! Training results saved to Drive.")
print("Contents:")
saved_path = '/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_b_augv2'
for item in os.listdir(saved_path):
    print(f"  {item}")


In [ ]:
#Resume Training if Stopped
from ultralytics import YOLO
from ultralytics.utils import callbacks
import shutil
import os

model = YOLO('/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_b_augv2_backup/best.pt')

model.train(
    data='/content/drive/MyDrive/FracDetectYolo/filtered_dataset/meta.yaml',
    resume=True,
    epochs=100,
    patience=15,
    batch=16,
    optimizer='SGD',
    lr0=0.001,
    lrf=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    warmup_epochs=5,
    cos_lr=True,
    cls=1.5,
    label_smoothing=0.1,
    device=0,
    project='fracture_detection',
    name='model_a_baseline',
    exist_ok=True,

    # Augmentation disabled
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    flipud=0.0,
    fliplr=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
)

In [ ]:
shutil.copytree(
    '/content/fracture_detection/model_a_baseline',
    '/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_a_baseline',
    dirs_exist_ok=True
)
print("Done! Training results saved to Drive.")
print("Contents:")
saved_path = '/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_a_baseline'
for item in os.listdir(saved_path):
    print(f"  {item}")

In [ ]:
# Using Model Weights on the Validation Set
from ultralytics import YOLO

model = YOLO('/content/fracture_detection/model_a_baseline/weights/best.pt')

metrics = model.val(
    data='/content/drive/MyDrive/FracDetectYolo/filtered_dataset/meta.yaml',
    split='val',
    device=0,
)

target_classes = [
    '23r-M/2.1',   # 0
    '23r-M/3.1',   # 1
    '23-M/3.1',    # 2
    '23-M/2.1',    # 3
    '23r-E/2.1',   # 4
    '22r-D/2.1',   # 5
    '23r-E/1',     # 6
    '22-D/2.1',    # 7
]

print(f"\n{'Class':<15} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'F1':>10}")
print("-" * 55)

for i, (p, r, ap) in enumerate(zip(metrics.box.p, metrics.box.r, metrics.box.ap50)):
    name = target_classes[i] if i < len(target_classes) else f'class_{i}'
    f1   = 2 * (p * r) / (p + r) if (p + r) > 0 else 0
    print(f"{name:<15} {p:>10.4f} {r:>10.4f} {ap:>10.4f} {f1:>10.4f}")

print("-" * 55)
print(f"{'Overall':<15} {metrics.box.mp:>10.4f} {metrics.box.mr:>10.4f} {metrics.box.map50:>10.4f}")

In [ ]:
import torch

# Store true original before any patching
_true_torch_load = torch.serialization.load

def _patched_load(f, map_location=None, pickle_module=None, **kwargs):
    kwargs['weights_only'] = False
    return _true_torch_load(f, map_location=map_location, pickle_module=pickle_module, **kwargs)

torch.load = _patched_load

from ultralytics import YOLO

WEIGHTS_PATH = '/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_a_augv2/weights/best.pt'
model = YOLO(WEIGHTS_PATH)

metrics = model.val(
    data='/content/model_a/meta.yaml',
    split='test',
    save_json=True,
    plots=True,
    device=0,
    project='/content/fracture_detection',
    name='model_a_augv2_test'
)

print(f"\n=== Augmented Test Set Results ===")
print(f"mAP@50:      {metrics.box.map50:.4f}")
print(f"mAP@50-95:   {metrics.box.map:.4f}")
print(f"Precision:   {metrics.box.mp:.4f}")
print(f"Recall:      {metrics.box.mr:.4f}")

print(f"\n=== Per-Class AP@50 ===")
names = [
    '23r-M/2.1', '23r-M/3.1', '23-M/3.1', '23-M/2.1',
    '23r-E/2.1', '22r-D/2.1', '23r-E/1',  '22-D/2.1',
]
for i, name in enumerate(names):
    print(f"  {name}: {metrics.box.ap50[i]:.4f}")

In [ ]:
#Save Testing Results
import shutil
shutil.copytree(
    '/content/fracture_detection/model_a_augv2_test',
    '/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_a_augv2_test',
    dirs_exist_ok=True
)
print("Training results saved to Drive!")

In [ ]:
#Check Results across different thresholds
from ultralytics import YOLO
import pandas as pd

model = YOLO('/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_b_aug/weights/best.pt')

results = []
for conf in [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.50]:
    metrics = model.val(
        data='/content/drive/MyDrive/FracDetectYolo/filtered_dataset/meta.yaml',
        split='test',
        conf=conf,
        iou=0.7,
        verbose=False
    )
    results.append({
        'conf': conf,
        'mAP50': round(metrics.box.map50, 4),
        'precision': round(metrics.box.mp, 4),
        'recall': round(metrics.box.mr, 4),
        'f1': round(2 * metrics.box.mp * metrics.box.mr /
                   (metrics.box.mp + metrics.box.mr + 1e-8), 4)
    })

df = pd.DataFrame(results)
print(df.to_string(index=False))

## Tuning the Model

In [ ]:
import random
from collections import defaultdict

random.seed(42)

# Sample balanced subset for tuning 50 per class
tune_cap = 50

class_to_stems = defaultdict(list)

for stem in train_stems:
    if '_aug_' in stem:
        continue  # skip augmented stems for tuning subset

    lbl_path = os.path.join(labels_dir, f"{stem}.txt")
    if not os.path.exists(lbl_path):
        continue
    with open(lbl_path) as f:
        line = f.read().strip()
    if not line:
        continue

    class_id = int(line.split()[0])
    class_to_stems[class_id].append(stem)

tune_stems = []
for cls_id, stems in sorted(class_to_stems.items()):
    sampled = random.sample(stems, min(tune_cap, len(stems)))
    tune_stems.extend(sampled)
    print(f"  Class {cls_id}: {len(stems)} available -> {len(sampled)} sampled")

# Add non-fractures
non_fracture_stems = []
for stem in train_stems:
    if '_aug_' in stem:
        continue
    lbl_path = os.path.join(labels_dir, f"{stem}.txt")
    if not os.path.exists(lbl_path):
        continue
    with open(lbl_path) as f:
        line = f.read().strip()
    if not line:  # empty = non-fracture
        non_fracture_stems.append(stem)

# Cap non-fractures to roughly match fracture count
sampled_nf = random.sample(non_fracture_stems, min(100, len(non_fracture_stems)))
tune_stems.extend(sampled_nf)
print(f"  Non-fracture: {len(non_fracture_stems)} available -> {len(sampled_nf)} sampled")
print(f"\nTotal tuning stems: {len(tune_stems)}")

#Get validation Stems
random.seed(42)
tune_val_stems = random.sample(val_stems, min(300, len(val_stems)))

print(f"Tune train stems: {len(tune_stems)}")
print(f"Tune val stems:   {len(tune_val_stems)}")

print(f"\nTotal tuning stems: {len(tune_stems)}")

In [ ]:
import random


In [ ]:
import os
import shutil

images_dir = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/full_dataset'
labels_dir = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/labels_model_u_remapped'

tune_img_train = '/content/model_u_tune/images/train'
tune_lbl_train = '/content/model_u_tune/labels/train'
tune_img_val   = '/content/model_u_tune/images/val'
tune_lbl_val   = '/content/model_u_tune/labels/val'

for d in [tune_img_train, tune_lbl_train, tune_img_val, tune_lbl_val]:
    os.makedirs(d, exist_ok=True)

def copy_stems(stems, img_dst, lbl_dst):
    copied = skipped = 0
    for stem in stems:
        img_src = os.path.join(images_dir, f"{stem}.png")
        lbl_src = os.path.join(labels_dir, f"{stem}.txt")
        if not os.path.exists(img_src) or not os.path.exists(lbl_src):
            skipped += 1
            continue
        shutil.copy2(img_src, os.path.join(img_dst, f"{stem}.png"))
        shutil.copy2(lbl_src, os.path.join(lbl_dst, f"{stem}.txt"))
        copied += 1
    return copied, skipped

copied, skipped = copy_stems(tune_stems, tune_img_train, tune_lbl_train)
print(f"Train — copied: {copied}, skipped: {skipped}")

copied, skipped = copy_stems(tune_val_stems, tune_img_val, tune_lbl_val)
print(f"Val   — copied: {copied}, skipped: {skipped}")

In [ ]:
import yaml

tune_meta = {
    'path':  '/content/model_u_tune/',
    'train': 'images/train',
    'val':   'images/val',
    'nc':    8,
    'names': {
        0: '23r-M/2.1',
        1: '23r-M/3.1',
        2: '23-M/3.1',
        3: '23-M/2.1',
        4: '23r-E/2.1',
        5: '22r-D/2.1',
        6: '23r-E/1',
        7: '22-D/2.1',
    }
}

tune_meta_path = '/content/drive/MyDrive/FracDetectYolo/filtered_dataset/meta_tune.yaml'
with open(tune_meta_path, 'w') as f:
    yaml.dump(tune_meta, f, default_flow_style=False, allow_unicode=True)

print(f"Written meta_tune.yaml")
print(f"  Train: {len(tune_stems)} images")
print(f"  Val:   {len(tune_val_stems)} images")

## Test Stuff

In [ ]:
# Test and Save all Model Results
import torch
import yaml
# Store true original before any patching
_true_torch_load = torch.serialization.load

def _patched_load(f, map_location=None, pickle_module=None, **kwargs):
    kwargs['weights_only'] = False
    return _true_torch_load(f, map_location=map_location, pickle_module=pickle_module, **kwargs)

torch.load = _patched_load

from ultralytics import YOLO

def write_meta(base_path, nc=8):
    meta = {
        'path': base_path,
        'train': 'images/train',
        'val':   'images/val',
        'test':  'images/test',
        'nc': nc,
        'names': {
            0: '23r-M/2.1',
            1: '23r-M/3.1',
            2: '23-M/3.1',
            3: '23-M/2.1',
            4: '23r-E/2.1',
            5: '22r-D/2.1',
            6: '23r-E/1',
            7: '22-D/2.1',
        }
    }
    meta_path = f'{base_path}/meta.yaml'
    with open(meta_path, 'w') as f:
        yaml.dump(meta, f, default_flow_style=False, allow_unicode=True)
    return meta_path

# Write once per dataset, outside the loop
meta_u = write_meta('/content/model_u')
meta_a = write_meta('/content/model_a')
meta_b = write_meta('/content/model_b')

models = {
    'model_u_baseline': ('/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_u_baseline_v5/weights/best.pt', meta_u),
    'model_u_aug':      ('/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_u_augmented/weights/best.pt',      meta_u),
    'model_a_baseline': ('/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_a_baseline/weights/best.pt', meta_a),
    'model_a_aug':      ('/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_a_aug/weights/best.pt',      meta_a),
    'model_b_baseline': ('/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_b_baseline/weights/best.pt', meta_b),
    'model_b_aug':      ('/content/drive/MyDrive/FracDetectYolo/FCE-YOLOv8/runs/model_b_aug/weights/best.pt',      meta_b),
}

all_results = {}
for run_name, (model_path, meta_path) in models.items():
    print(f"\nEvaluating {run_name}...")
    model = YOLO(model_path)
    metrics = model.val(
        data=meta_path,
        split='test',
        verbose=True
    )
    all_results[run_name] = metrics

In [ ]:
# Save All Results
import pandas as pd

class_names = [
    '23r-M/2.1', '23r-M/3.1', '23-M/3.1', '23-M/2.1',
    '23r-E/2.1', '22r-D/2.1', '23r-E/1', '22-D/2.1'
]

rows = []
for run_name, metrics in all_results.items():
    row = {'model': run_name, 'class': 'all',
           'mAP50': round(metrics.box.map50, 4),
           'precision': round(metrics.box.mp, 4),
           'recall': round(metrics.box.mr, 4)}
    rows.append(row)
    for i, ap50 in enumerate(metrics.box.ap50):
        rows.append({'model': run_name, 'class': class_names[i],
                     'mAP50': round(ap50, 4),
                     'precision': None, 'recall': None})

pd.DataFrame(rows).to_csv(
    '/content/drive/MyDrive/FracDetectYolo/all_results.csv', index=False
)
print("Saved.")

In [ ]:
# Creates Table of all Results
import pandas as pd

def extract_results_table(all_results, class_names):
    rows = []

    for run_name, metrics in all_results.items():
        rows.append({
            'model': run_name,
            'class': 'all',
            'mAP50': round(float(metrics.box.map50), 4),
            'mAP50-95': round(float(metrics.box.map), 4),
            'precision': round(float(metrics.box.mp), 4),
            'recall': round(float(metrics.box.mr), 4),
        })

        for i, (ap50, ap5095) in enumerate(zip(metrics.box.ap50, metrics.box.ap)):
            rows.append({
                'model': run_name,
                'class': class_names[i],
                'mAP50': round(float(ap50), 4),
                'mAP50-95': round(float(ap5095), 4),
                'precision': None,
                'recall': None,
            })

    return pd.DataFrame(rows)

class_names = [
    '23r-M/2.1', '23r-M/3.1', '23-M/3.1', '23-M/2.1',
    '23r-E/2.1', '22r-D/2.1', '23r-E/1', '22-D/2.1'
]

df = extract_results_table(all_results, class_names)

# Wide format pivot — models as columns, classes as rows
pivot = df.pivot_table(
    index='class',
    columns='model',
    values='mAP50'
).round(4)

# Put 'all' row first
pivot = pd.concat([
    pivot.loc[['all']],
    pivot.drop('all')
])

print(pivot.to_string())